# Conv1D Autoencoder + VAE — Reconstruction-Based Detector

**Replaces LSTM-AE** in the ensemble. **Tag:** `conv_ae`

- **Credit Card:** VAE with `reduce_mean` loss (numerically stable)
- **NAB:** 1D-CNN Autoencoder with fixed output shape handling


## Cell 0 — Setup

In [1]:
# Cell 0 — Setup
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, glob, json, warnings, urllib.request
import numpy as np
import pandas as pd
import joblib

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    precision_recall_curve,
)

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

RUNID = 'ensemble_run_001'
DRIVEROOT = '/content/drive/MyDrive/tsad_ensemble_runs'
NOTEBOOKTAG = 'conv_ae'

RUNDIR = os.path.join(DRIVEROOT, RUNID, NOTEBOOKTAG)
ARTIFACTDIR = os.path.join(RUNDIR, 'artifacts')
PREDICTIONSDIR = os.path.join(RUNDIR, 'predictions')
CACHEDIR = os.path.join(DRIVEROOT, '_cache')

for d in [ARTIFACTDIR, PREDICTIONSDIR, CACHEDIR]:
    os.makedirs(d, exist_ok=True)

print('RUNDIR       :', RUNDIR)
print('ARTIFACTDIR  :', ARTIFACTDIR)
print('PREDICTIONSDIR:', PREDICTIONSDIR)
print('TF version   :', tf.__version__)


Mounted at /content/drive
RUNDIR       : /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/conv_ae
ARTIFACTDIR  : /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/conv_ae/artifacts
PREDICTIONSDIR: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/conv_ae/predictions
TF version   : 2.19.0


## Cell 1 — Dataset Paths

In [2]:
# Cell 1 — Dataset Paths

MYDRIVE = '/content/drive/MyDrive'
CREDITCARDPATH = os.path.join(MYDRIVE, 'creditcard.csv')

NAB_CANDIDATES = [
    os.path.join(MYDRIVE, 'NAB Dataset'),
    os.path.join(MYDRIVE, 'NAB'),
    os.path.join(MYDRIVE, 'datasets', 'NAB Dataset'),
    os.path.join(MYDRIVE, 'datasets', 'NAB'),
]

def resolve_nab_root(candidates):
    for c in candidates:
        if os.path.isdir(c):
            csvs = glob.glob(os.path.join(c, '**', '*.csv'), recursive=True)
            if len(csvs) > 0:
                return c
    return None

NABROOT = resolve_nab_root(NAB_CANDIDATES)

NABLABELSLOCAL = os.path.join(CACHEDIR, 'nab_combined_windows.json')
NABLABELSURL = 'https://raw.githubusercontent.com/numenta/NAB/master/labels/combined_windows.json'
if not os.path.exists(NABLABELSLOCAL):
    urllib.request.urlretrieve(NABLABELSURL, NABLABELSLOCAL)
with open(NABLABELSLOCAL) as f:
    NABWINDOWSMAP = json.load(f)

assert os.path.exists(CREDITCARDPATH), f'creditcard.csv not found at {CREDITCARDPATH}'
assert NABROOT is not None, 'NAB root not found.'

nab_csvs = [f for f in glob.glob(os.path.join(NABROOT, '**', '*.csv'), recursive=True) if 'README' not in f]
print(f'Credit Card : {CREDITCARDPATH}')
print(f'NAB root    : {NABROOT} ({len(nab_csvs)} CSVs)')
print(f'NAB labels  : {len(NABWINDOWSMAP)} entries')


Credit Card : /content/drive/MyDrive/creditcard.csv
NAB root    : /content/drive/MyDrive/NAB Dataset (58 CSVs)
NAB labels  : 58 entries


## Cell 2 — Shared Utilities

In [3]:
# Cell 2 — Shared Utilities

def keep_runs(y, min_len=3):
    y = np.asarray(y, dtype=np.int8).copy()
    n = len(y)
    i = 0
    while i < n:
        if y[i] == 1:
            j = i
            while j < n and y[j] == 1:
                j += 1
            if (j - i) < min_len:
                y[i:j] = 0
            i = j
        else:
            i += 1
    return y

def point_adjust(y_true, y_pred):
    yt = np.asarray(y_true, dtype=np.int8)
    yp = np.asarray(y_pred, dtype=np.int8).copy()
    n = len(yt)
    i = 0
    while i < n:
        if yt[i] == 1:
            j = i
            while j < n and yt[j] == 1:
                j += 1
            if yp[i:j].any():
                yp[i:j] = 1
            i = j
        else:
            i += 1
    return yp

def best_f1_threshold(y, scores, ngrid=300, qlo=0.50, qhi=0.999, min_run=0):
    y = np.asarray(y, dtype=int)
    scores = np.asarray(scores, dtype=float)
    if y.sum() == 0:
        return float(np.percentile(scores, 99.5)), 0.0
    qs = np.linspace(qlo, qhi, ngrid)
    thr_list = np.unique(np.quantile(scores, qs))
    best_t, best_f = float(thr_list[-1]), -1.0
    for t in thr_list:
        p = (scores >= t).astype(int)
        if min_run > 0:
            p = keep_runs(p, min_len=min_run)
        f = f1_score(y, p, zero_division=0)
        if f > best_f:
            best_f, best_t = float(f), float(t)
    return best_t, best_f

def compute_metrics(y, pred, scores=None, prefix=''):
    m = {
        f'{prefix}precision': float(precision_score(y, pred, zero_division=0)),
        f'{prefix}recall': float(recall_score(y, pred, zero_division=0)),
        f'{prefix}f1': float(f1_score(y, pred, zero_division=0)),
    }
    if scores is not None and len(np.unique(y)) == 2:
        m[f'{prefix}rocauc'] = float(roc_auc_score(y, scores))
        m[f'{prefix}prauc'] = float(average_precision_score(y, scores))
    else:
        m[f'{prefix}rocauc'] = float('nan')
        m[f'{prefix}prauc'] = float('nan')
    return m

print('Utilities ready.')


Utilities ready.


## Cell 3 — Credit Card VAE

**v2 fixes:**
- `reduce_mean` instead of `reduce_sum` to prevent NaN from large feature dimension
- `mode='min'` on EarlyStopping/ReduceLROnPlateau for custom metric names
- Clip scaled features to [-5, 5]

In [4]:
# Cell 3 — Credit Card Deep Autoencoder (replaces VAE for better CC F1)
# RATIONALE: VAE's KL divergence term over-regularizes the latent space,
# preventing the model from learning a tight normal-data boundary.
# A standard deep AE with dropout achieves better anomaly separation.

print('Loading Credit Card data...')
df = pd.read_csv(CREDITCARDPATH).dropna().reset_index(drop=True)
y_all = df['Class'].astype(int).values

X_df = df.copy()
X_df['hoursin'] = np.sin(2 * np.pi * X_df['Time'] / 86400.0)
X_df['hourcos'] = np.cos(2 * np.pi * X_df['Time'] / 86400.0)
X_df['Amountlog'] = np.log1p(X_df['Amount'])
X_df['AmountSq'] = X_df['Amount'] ** 2
for v in ['V1', 'V3', 'V4', 'V7', 'V10', 'V12', 'V14', 'V17']:
    X_df[f'{v}_abs'] = X_df[v].abs()

feature_cols = (
    [f'V{i}' for i in range(1, 29)]
    + ['Amountlog', 'AmountSq', 'hoursin', 'hourcos']
    + [f'{v}_abs' for v in ['V1', 'V3', 'V4', 'V7', 'V10', 'V12', 'V14', 'V17']]
)
X_all = X_df[feature_cols].values.astype(np.float64)

# Same split: 60/20/20
idx = np.arange(len(y_all), dtype=np.int64)
idx_tune, idx_test = train_test_split(idx, test_size=0.20, stratify=y_all, random_state=RANDOM_STATE)
idx_train, idx_val = train_test_split(idx_tune, test_size=0.25, stratify=y_all[idx_tune], random_state=RANDOM_STATE)

X_train_normal = X_all[idx_train][y_all[idx_train] == 0]
X_val = X_all[idx_val]
y_val = y_all[idx_val]
X_test = X_all[idx_test]
y_test = y_all[idx_test]

# Scale + clip
cc_scaler = RobustScaler()
X_train_normal_sc = np.clip(cc_scaler.fit_transform(X_train_normal), -5, 5)
X_val_sc = np.clip(cc_scaler.transform(X_val), -5, 5)
X_test_sc = np.clip(cc_scaler.transform(X_test), -5, 5)

print(f'Train normal: {len(X_train_normal):,} rows')
print(f'Validation  : {len(X_val):,} rows ({y_val.sum()} frauds)')
print(f'Test        : {len(X_test):,} rows ({y_test.sum()} frauds)')

# --- Build Deep AE ---
input_dim = X_train_normal_sc.shape[1]
print(f'Input dim: {input_dim}')

def build_deep_ae(input_dim, bottleneck=6):
    inp = keras.Input(shape=(input_dim,))
    x = layers.Dense(128, activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(16, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    encoded = layers.Dense(bottleneck, activation='relu', name='bottleneck')(x)
    x = layers.Dense(16, activation='relu')(encoded)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    decoded = layers.Dense(input_dim, activation='linear')(x)
    return keras.Model(inp, decoded, name='deep_ae')

ae_model = build_deep_ae(input_dim, bottleneck=6)
ae_model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='mse')
ae_model.summary()

print('\nTraining Deep AE on normal data only...')
ae_history = ae_model.fit(
    X_train_normal_sc, X_train_normal_sc,
    epochs=150, batch_size=512, validation_split=0.1,
    callbacks=[
        callbacks.EarlyStopping(patience=15, restore_best_weights=True, monitor='val_loss'),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=7, min_lr=1e-6),
    ],
    verbose=1,
)
print(f'Deep AE training complete. Best val loss: {min(ae_history.history["val_loss"]):.6f}')


Loading Credit Card data...
Train normal: 170,588 rows
Validation  : 56,962 rows (99 frauds)
Test        : 56,962 rows (98 frauds)
Input dim: 40


Model: "deep_ae"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 40)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         5,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bottleneck (Dense)              │ (None, 6)              │           102 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 34,382 (134.30 KB)

 Trainable params: 33,422 (130.55 KB)

 Non-trainable params: 960 (3.75 KB)


Training Deep AE on normal data only...
Epoch 1/150
300/300 ━━━━━━━━━━━━━━━━━━━━ 20s 28ms/step - loss: 0.9416 - val_loss: 0.6840 - learning_rate: 0.0010
Epoch 2/150
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.7002 - val_loss: 0.6010 - learning_rate: 0.0010
Epoch 3/150
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.6459 - val_loss: 0.5588 - learning_rate: 0.0010
Epoch 4/150
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.6122 - val_loss: 0.5253 - learning_rate: 0.0010
Epoch 5/150
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.5836 - val_loss: 0.4980 - learning_rate: 0.0010
Epoch 6/150
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.5629 - val_loss: 0.4771 - learning_rate: 0.0010
Epoch 7/150
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.5457 - val_loss: 0.4573 - learning_rate: 0.0010
Epoch 8/150
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.5307 - val_loss: 0.4387 - learning_rate: 0.0010
Epoch 9/150
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.5183 -

## Cell 4 — Credit Card VAE Scoring & Evaluation

In [5]:
# Cell 4 — CC Deep AE Scoring (multi-signal)

def ae_anomaly_score(ae_model, X_sc):
    recon = ae_model.predict(X_sc, batch_size=4096, verbose=0)
    errors = X_sc - recon
    mse = np.mean(errors ** 2, axis=1)
    mae = np.mean(np.abs(errors), axis=1)
    max_err = np.max(np.abs(errors), axis=1)
    combined = 0.50 * mse + 0.30 * mae + 0.20 * max_err
    return combined.astype(np.float32)

scores_val = ae_anomaly_score(ae_model, X_val_sc)
scores_test = ae_anomaly_score(ae_model, X_test_sc)

prec, rec, thr = precision_recall_curve(y_val, scores_val)
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
if len(thr) > 0:
    best_idx = int(np.nanargmax(f1s))
    cc_threshold = float(thr[best_idx])
    print(f'PR-curve threshold: {cc_threshold:.6f} (val F1={f1s[best_idx]:.4f})')
else:
    cc_threshold = float(np.percentile(scores_val, 99.5))
    print(f'Fallback threshold: {cc_threshold:.6f}')

cc_preds = (scores_test >= cc_threshold).astype(np.int8)
cc_ytrue = y_test.astype(np.int8)

strict = compute_metrics(cc_ytrue, cc_preds, scores_test)

cc_results = {
    'dataset': 'creditcard',
    'protocol': 'semi-supervised strict holdout (20% test)',
    **strict,
    'threshold': cc_threshold,
    'n_anomalies_true': int(cc_ytrue.sum()),
    'n_anomalies_pred': int(cc_preds.sum()),
}

print()
print('=' * 60)
print('CREDIT CARD DEEP AE RESULTS')
print('=' * 60)
for k, v in cc_results.items():
    print(f'  {k:25s}: {v:.6f}' if isinstance(v, float) else f'  {k:25s}: {v}')

# DEBUG — Deep AE diagnostics
print()
print('=== DEEP AE DIAGNOSTICS ===')
print(f'Normal test MSE mean: {np.mean((X_test_sc[y_test==0] - ae_model.predict(X_test_sc[y_test==0], verbose=0))**2, axis=1).mean():.6f}')
print(f'Fraud test MSE mean:  {np.mean((X_test_sc[y_test==1] - ae_model.predict(X_test_sc[y_test==1], verbose=0))**2, axis=1).mean():.6f}')
fp = int(((cc_preds == 1) & (cc_ytrue == 0)).sum())
fn = int(((cc_preds == 0) & (cc_ytrue == 1)).sum())
tp = int(((cc_preds == 1) & (cc_ytrue == 1)).sum())
print(f'Confusion: TP={tp} FP={fp} FN={fn}')


PR-curve threshold: 3.511863 (val F1=0.6806)

CREDIT CARD DEEP AE RESULTS
  dataset                  : creditcard
  protocol                 : semi-supervised strict holdout (20% test)
  precision                : 0.721649
  recall                   : 0.714286
  f1                       : 0.717949
  rocauc                   : 0.957455
  prauc                    : 0.687445
  threshold                : 3.511863
  n_anomalies_true         : 98
  n_anomalies_pred         : 97

=== DEEP AE DIAGNOSTICS ===
Normal test MSE mean: 0.245639
Fraud test MSE mean:  5.760892
Confusion: TP=70 FP=27 FN=28


## Cell 5 — NAB Conv1D Autoencoder

**v2 fix:** Replaced broken Cropping1D with a Lambda layer that slices output to
match WINDOW_SIZE exactly. WINDOW_SIZE=32 (power of 2) avoids pool/upsample mismatch.

In [6]:
# Cell 5 — NAB Conv1D Autoencoder

class NABConvAEAgent:
    WINDOW_SIZE = 32       # power of 2 => clean pool/upsample
    STRIDE = 1
    MIN_RUN = 3
    TRAIN_FRAC = 0.40
    VAL_FRAC = 0.30
    EPOCHS = 60
    BATCH_SIZE = 64

    def __init__(self):
        self.series_models = {}

    def match_nab_key(self, csv_path):
        rel = os.path.relpath(csv_path, NABROOT).replace('\\', '/')
        if rel in NABWINDOWSMAP:
            return rel
        base = os.path.basename(csv_path)
        for k in NABWINDOWSMAP:
            if os.path.basename(k) == base:
                return k
        return None

    def make_labels(self, timestamps, csv_path):
        y = np.zeros(len(timestamps), dtype=np.int8)
        key = self.match_nab_key(csv_path)
        if key is None or not NABWINDOWSMAP.get(key):
            return y
        for t0, t1 in NABWINDOWSMAP[key]:
            start = pd.Timestamp(t0)
            end = pd.Timestamp(t1)
            y[(timestamps >= start) & (timestamps <= end)] = 1
        return y

    def extract_features(self, values):
        s = pd.Series(values.astype(np.float64))
        parts = [s.rename('value')]
        for w in [10, 30]:
            roll = s.rolling(w, min_periods=1)
            parts.extend([
                roll.mean().rename(f'rmean_{w}'),
                roll.std().fillna(0).rename(f'rstd_{w}'),
                (roll.max() - roll.min()).rename(f'rrange_{w}'),
            ])
        parts.append(s.diff(1).fillna(0).rename('diff1'))
        rmean = s.rolling(30, min_periods=1).mean()
        rstd = s.rolling(30, min_periods=1).std().fillna(1) + 1e-9
        parts.append(((s - rmean) / rstd).rename('zscore_30'))
        df = pd.concat(parts, axis=1)
        return np.nan_to_num(df.values, nan=0.0, posinf=0.0, neginf=0.0)

    def make_windows(self, features):
        n = len(features)
        w = self.WINDOW_SIZE
        if n < w + 1:
            return np.empty((0, w, features.shape[1]))
        windows = np.array([features[i:i+w] for i in range(0, n - w + 1, self.STRIDE)])
        return windows

    def build_conv1d_ae(self, n_features):
        """Build 1D-CNN autoencoder.
        WINDOW_SIZE=32: pool(2)->16, pool(2)->8, upsample(2)->16, upsample(2)->32
        Output matches input exactly."""
        inp = keras.Input(shape=(self.WINDOW_SIZE, n_features))
        # Encoder
        x = layers.Conv1D(32, 3, activation='relu', padding='same')(inp)
        x = layers.MaxPooling1D(2)(x)      # 32 -> 16
        x = layers.Conv1D(16, 3, activation='relu', padding='same')(x)
        x = layers.MaxPooling1D(2)(x)      # 16 -> 8
        # Bottleneck
        x = layers.Conv1D(8, 3, activation='relu', padding='same')(x)
        # Decoder
        x = layers.Conv1D(16, 3, activation='relu', padding='same')(x)
        x = layers.UpSampling1D(2)(x)      # 8 -> 16
        x = layers.Conv1D(32, 3, activation='relu', padding='same')(x)
        x = layers.UpSampling1D(2)(x)      # 16 -> 32
        x = layers.Conv1D(n_features, 3, activation='linear', padding='same')(x)
        model = keras.Model(inp, x, name='conv1d_ae')
        model.compile(optimizer='adam', loss='mse')
        return model

    def score_windows(self, model, windows):
        if len(windows) == 0:
            return np.array([])
        recon = model.predict(windows, batch_size=256, verbose=0)
        mse = np.mean((windows - recon) ** 2, axis=(1, 2))
        return mse

    def window_scores_to_pointwise(self, window_scores, n_total):
        point_scores = np.zeros(n_total, dtype=np.float64)
        w = self.WINDOW_SIZE
        for i, score in enumerate(window_scores):
            start = i * self.STRIDE
            end = min(start + w, n_total)
            point_scores[start:end] = np.maximum(point_scores[start:end], score)
        if len(window_scores) > 0:
            point_scores[:w] = np.where(point_scores[:w] > 0, point_scores[:w], window_scores[0])
        return point_scores

    def fit(self):
        all_csv = sorted(glob.glob(os.path.join(NABROOT, '**', '*.csv'), recursive=True))
        all_csv = [f for f in all_csv if 'README' not in f]
        trained = 0
        for csv_path in all_csv:
            try:
                dfn = pd.read_csv(csv_path, parse_dates=['timestamp'])
                values = dfn['value'].values
                timestamps = dfn['timestamp'].values
                y = self.make_labels(timestamps, csv_path)

                n = len(values)
                n_train = int(n * self.TRAIN_FRAC)

                features = self.extract_features(values)

                scaler = RobustScaler()
                scaler.fit(features[:n_train])
                features_sc = scaler.transform(features)

                train_features = features_sc[:n_train]
                train_y = y[:n_train]
                train_normal = train_features[train_y == 0] if train_y.sum() > 0 else train_features

                train_windows = self.make_windows(train_normal)
                if len(train_windows) < 10:
                    continue

                n_features = features.shape[1]
                model = self.build_conv1d_ae(n_features)
                model.fit(
                    train_windows, train_windows,
                    epochs=self.EPOCHS,
                    batch_size=self.BATCH_SIZE,
                    validation_split=0.15,
                    callbacks=[
                        callbacks.EarlyStopping(patience=8, restore_best_weights=True),
                    ],
                    verbose=0,
                )

                rel_key = self.match_nab_key(csv_path)
                self.series_models[rel_key] = {
                    'model': model,
                    'scaler': scaler,
                    'n_features': n_features,
                }
                trained += 1

            except Exception as e:
                print(f'  [skip] {os.path.basename(csv_path)}: {e}')

        print(f'Trained Conv1D-AE on {trained} / {len(all_csv)} NAB series.')

    def evaluate(self):
        all_csv = sorted(glob.glob(os.path.join(NABROOT, '**', '*.csv'), recursive=True))
        all_csv = [f for f in all_csv if 'README' not in f]

        bundle = {}
        S_all, Y_all, P_all, PA_all = [], [], [], []
        category_results = {}

        for csv_path in all_csv:
            try:
                rel_key = self.match_nab_key(csv_path)
                if rel_key not in self.series_models:
                    continue

                info = self.series_models[rel_key]
                model = info['model']
                scaler = info['scaler']

                dfn = pd.read_csv(csv_path, parse_dates=['timestamp'])
                values = dfn['value'].values
                timestamps = dfn['timestamp'].values
                y = self.make_labels(timestamps, csv_path)
                n = len(values)

                n_train = int(n * self.TRAIN_FRAC)
                n_val = int(n * self.VAL_FRAC)
                test_start = n_train + n_val

                features = self.extract_features(values)
                features_sc = scaler.transform(features)

                all_windows = self.make_windows(features_sc)
                if len(all_windows) == 0:
                    continue

                all_window_scores = self.score_windows(model, all_windows)
                point_scores = self.window_scores_to_pointwise(all_window_scores, n)

                s_v = point_scores[n_train:n_train + n_val]
                y_v = y[n_train:n_train + n_val]

                s_h = point_scores[test_start:].astype(np.float32)
                y_h = y[test_start:].astype(np.int8)

                if len(s_h) == 0:
                    continue

                thr_val, _ = best_f1_threshold(y_v, s_v, min_run=self.MIN_RUN)

                p_h = (s_h >= thr_val).astype(np.int8)
                p_h = keep_runs(p_h, min_len=self.MIN_RUN)
                pa_h = point_adjust(y_h, p_h)

                entity_id = rel_key.replace('/', '_').replace('.csv', '')
                bundle[entity_id] = {
                    'entityid': entity_id, 'nabkey': rel_key,
                    'scoresfull': s_h, 'yfull': y_h,
                    'predfull': p_h, 'pa_predfull': pa_h,
                    'threshold': float(thr_val),
                }

                S_all.append(s_h)
                Y_all.append(y_h)
                P_all.append(p_h)
                PA_all.append(pa_h)

                cat = rel_key.split('/')[0] if '/' in rel_key else 'unknown'
                if cat not in category_results:
                    category_results[cat] = {'S': [], 'Y': [], 'P': [], 'PA': []}
                category_results[cat]['S'].append(s_h)
                category_results[cat]['Y'].append(y_h)
                category_results[cat]['P'].append(p_h)
                category_results[cat]['PA'].append(pa_h)

            except Exception as e:
                print(f'  [eval skip] {csv_path}: {e}')

        if not S_all:
            return {}, bundle, category_results

        S = np.concatenate(S_all)
        Y = np.concatenate(Y_all)
        P = np.concatenate(P_all)
        PA = np.concatenate(PA_all)

        strict = compute_metrics(Y, P, S, prefix='strict_')
        pa_metrics = compute_metrics(Y, PA, S, prefix='pa_')

        results = {
            'dataset': 'nab',
            'protocol': f'temporal holdout: train {int(100*self.TRAIN_FRAC)}% / val {int(100*self.VAL_FRAC)}% / test {int(100*(1-self.TRAIN_FRAC-self.VAL_FRAC))}%',
            **strict, **pa_metrics,
            'n_anomalies_true': int(Y.sum()),
            'n_anomalies_pred': int(P.sum()),
            'n_series': len(bundle),
        }

        return results, bundle, category_results

print('NABConvAEAgent defined.')
nabagent = NABConvAEAgent()
print('Fitting NAB Conv1D-AE...')
nabagent.fit()


NABConvAEAgent defined.
Fitting NAB Conv1D-AE...
Trained Conv1D-AE on 58 / 58 NAB series.


## Cell 6 — NAB Evaluation & Per-Category Breakdown

In [7]:
# Cell 6 — NAB Evaluation

print('Evaluating NAB Conv1D-AE...')
nab_results, nab_bundle, nab_cat_results = nabagent.evaluate()

print()
print('=' * 60)
print('NAB CONV1D-AE RESULTS (GLOBAL)')
print('=' * 60)
for k, v in nab_results.items():
    print(f'  {k:25s}: {v:.6f}' if isinstance(v, float) else f'  {k:25s}: {v}')

print()
print('NAB PER-CATEGORY RESULTS')
print('=' * 90)
cat_rows = []
for cat, data in sorted(nab_cat_results.items()):
    Y = np.concatenate(data['Y'])
    P = np.concatenate(data['P'])
    S = np.concatenate(data['S'])
    PA = np.concatenate(data['PA'])

    strict = compute_metrics(Y, P, S)
    pa = compute_metrics(Y, PA, S, prefix='pa_')

    row = {
        'category': cat,
        'n_series': len(data['Y']),
        'n_anomalies': int(Y.sum()),
        'f1': strict['f1'],
        'pa_f1': pa['pa_f1'],
        'rocauc': strict.get('rocauc', float('nan')),
    }
    cat_rows.append(row)
    print(f"  {cat:30s}  series={len(data['Y']):3d}  "
          f"strict F1={strict['f1']:.4f}  PA F1={pa['pa_f1']:.4f}")

cat_df = pd.DataFrame(cat_rows)
display(cat_df)


Evaluating NAB Conv1D-AE...



NAB CONV1D-AE RESULTS (GLOBAL)
  dataset                  : nab
  protocol                 : temporal holdout: train 40% / val 30% / test 30%
  strict_precision         : 0.198147
  strict_recall            : 0.374260
  strict_f1                : 0.259111
  strict_rocauc            : 0.577547
  strict_prauc             : 0.130839
  pa_precision             : 0.349826
  pa_recall                : 0.814899
  pa_f1                    : 0.489511
  pa_rocauc                : 0.577547
  pa_prauc                 : 0.130839
  n_anomalies_true         : 11826
  n_anomalies_pred         : 22337
  n_series                 : 58

NAB PER-CATEGORY RESULTS
  artificialNoAnomaly             series=  5  strict F1=0.0000  PA F1=0.0000
  artificialWithAnomaly           series=  6  strict F1=0.2558  PA F1=0.3329
  realAWSCloudwatch               series= 17  strict F1=0.2598  PA F1=0.3790
  realAdExchange                  series=  6  strict F1=0.3080  PA F1=0.3227
  realKnownCause                  series=

,category,n_series,n_anomalies,f1,pa_f1,rocauc
0,artificialNoAnomaly,5,0,0.000000,0.000000,NaN
1,artificialWithAnomaly,6,1684,0.255784,0.332928,0.459886
2,realAWSCloudwatch,17,1916,0.259761,0.378955,0.468798
3,realAdExchange,6,259,0.307982,0.322741,0.802295
4,realKnownCause,7,3910,0.245560,0.698171,0.664604
5,realTraffic,7,1015,0.566347,0.784693,0.644189
6,realTweets,10,3042,0.233969,0.485555,0.618722


## Cell 7 — Coordinator-Compatible Export

In [8]:
# Cell 7 — Export

exported = {}
summaryrows = []

# --- Credit Card ---
cc_bundle_export = {
    'dataset': 'creditcard',
    'model': 'conv_ae_deep',
    'protocol': 'semi-supervised strict holdout (20% test)',
    'entities': {
        'creditcard': {
            'entityid': 'creditcard',
            'scoresfull': scores_test.astype(np.float32),
            'yfull': cc_ytrue,
            'predfull': cc_preds,
            'rowid': np.arange(len(cc_ytrue), dtype=np.int64),
            'originalrowid': idx_test.astype(np.int64),
            'threshold': cc_threshold,
        }
    }
}
cc_path = os.path.join(PREDICTIONSDIR, 'conv_ae_creditcard_strict.joblib')
joblib.dump(cc_bundle_export, cc_path)
exported['creditcard'] = cc_path
summaryrows.append({
    'dataset': 'creditcard',
    **{k: v for k, v in cc_results.items() if k != 'dataset'},
})
print('Saved', cc_path)

# --- NAB ---
nab_payload = {
    'dataset': 'nab',
    'model': 'conv_ae_conv1d',
    'protocol': nab_results.get('protocol', 'temporal holdout'),
    'entities': nab_bundle,
}
nab_path = os.path.join(PREDICTIONSDIR, 'conv_ae_nab_strict.joblib')
joblib.dump(nab_payload, nab_path)
exported['nab'] = nab_path
summaryrows.append({
    'dataset': 'nab',
    **{k: v for k, v in nab_results.items() if k != 'dataset'},
})
print('Saved', nab_path)

# --- Summary CSV ---
summary = pd.DataFrame(summaryrows)
summary_path = os.path.join(PREDICTIONSDIR, 'conv_ae_summary.csv')
summary.to_csv(summary_path, index=False)
print('Saved', summary_path)
display(summary)

# --- Manifest ---
manifest = {
    'runid': RUNID, 'driveroot': DRIVEROOT, 'notebooktag': NOTEBOOKTAG,
    'modelfamily': 'conv_ae', 'exportprotocol': 'ensembleexportv2',
    'artifactsdir': ARTIFACTDIR, 'predictionsdir': PREDICTIONSDIR,
    'exports': exported, 'summarycsv': summary_path,
    'creditcard_originalrowid_included': True,
    'datasets': ['creditcard', 'nab'], 'smd_dropped': True,
    'replaces': 'lstm_ae',
    'components': {'creditcard': 'variational_autoencoder', 'nab': 'conv1d_autoencoder'},
}
manifest_path = os.path.join(PREDICTIONSDIR, 'conv_ae_manifest.json')
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
print('Saved', manifest_path)

print()
print('Coordinator-ready files are in:', PREDICTIONSDIR)
print('Exported datasets:', sorted(exported.keys()))
print('Done.')


Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/conv_ae/predictions/conv_ae_creditcard_strict.joblib
Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/conv_ae/predictions/conv_ae_nab_strict.joblib
Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/conv_ae/predictions/conv_ae_summary.csv


,dataset,protocol,precision,recall,f1,rocauc,prauc,threshold,n_anomalies_true,n_anomalies_pred,...,strict_recall,strict_f1,strict_rocauc,strict_prauc,pa_precision,pa_recall,pa_f1,pa_rocauc,pa_prauc,n_series
0,creditcard,semi-supervised strict holdout (20% test),0.721649,0.714286,0.717949,0.957455,0.687445,3.511863,98,97,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,nab,temporal holdout: train 40% / val 30% / test 30%,NaN,NaN,NaN,NaN,NaN,NaN,11826,22337,...,0.37426,0.259111,0.577547,0.130839,0.349826,0.814899,0.489511,0.577547,0.130839,58.0


Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/conv_ae/predictions/conv_ae_manifest.json

Coordinator-ready files are in: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/conv_ae/predictions
Exported datasets: ['creditcard', 'nab']
Done.
